# Interactive napari viewer — cell type of interest selection

Same setup as 06_napari_cell_type.ipynb (morphology + H&E + cell boundary polygons), plus a checklist dock widget to isolate one or more `cell_type`s of interest — unselected cell types are hidden from the boundaries layer instead of being rebuilt from scratch.

In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Qt/OpenGL settings for Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"

import napari
import spatialdata as sd

In [2]:
PROJECT = Path.home() / "Projects/xenium_lung"
SDATA_PATH = PROJECT / "data/spatialdata/xenium_prime_5k_processed_annotated.zarr"

print("SpatialData path:", SDATA_PATH)
print("Exists:", SDATA_PATH.exists())

SpatialData path: /home/duydao/Projects/xenium_lung/data/spatialdata/xenium_prime_5k_processed_annotated.zarr
Exists: True


## Load the SpatialData object

In [3]:
sdata = sd.read_zarr(SDATA_PATH)
sdata

no parent found for <ome_zarr.reader.Label object at 0x7f6808341100>: None
no parent found for <ome_zarr.reader.Label object at 0x7f68082fde20>: None


SpatialData object, with associated Zarr store: /home/duydao/Projects/xenium_lung/data/spatialdata/xenium_prime_5k_processed_annotated.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 43270, 26720), (3, 21635, 13360), (3, 10817, 6680), (3, 5408, 3340), (3, 2704, 1670)
│     └── 'morphology_focus': DataTree[cyx] (4, 37348, 54086), (4, 18674, 27043), (4, 9337, 13521), (4, 4668, 6760), (4, 2334, 3380)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (37348, 54086), (18674, 27043), (9337, 13521), (4668, 6760), (2334, 3380)
│     └── 'nucleus_labels': DataTree[yx] (37348, 54086), (18674, 27043), (9337, 13521), (4668, 6760), (2334, 3380)
├── Points
│     └── 'transcripts': DataFrame with shape: (177464221, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (278328, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (277935, 3) (2D shapes)
└── Tables
      └── 'table': AnnData (266179, 4860)
with coordinate systems:
    ▸ 'global', with elements

## Extract the cell table (cell_type, spatial coordinates)

In [4]:
adata = sdata.tables["table"]

print("Cells:", adata.n_obs)
print("cell_type categories:")
print(adata.obs["cell_type"].value_counts())

Cells: 266179
cell_type categories:
cell_type
EC                  41590
Tumor_epithelial    38618
Fibroblast          33666
Treg                31717
Macrophage          25977
SMC/MC              15938
Neutrophil          14954
Ciliated            11179
T_cytotoxic         10247
Plasma               9999
B_cell               9740
AT1                  8916
AT2                  8752
Mast                 3902
PNECs                 984
Name: count, dtype: int64


In [5]:
# obsm["spatial"] is in the same raw (micron) units as the shapes/points elements —
# it needs the same scale transform as cell_boundaries to align with the pixel-space images.
from spatialdata.transformations import get_transformation

boundaries_transform = get_transformation(sdata.shapes["cell_boundaries"], to_coordinate_system="global")
scale_matrix = boundaries_transform.to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))

# Micron size of one pixel in the shared "global" pixel space (all layers use this same factor).
PIXEL_SIZE_UM = 1 / scale_matrix[0, 0]

print(scale_matrix)
print("Pixel size:", PIXEL_SIZE_UM, "µm/px")

[[4.70588235 0.         0.        ]
 [0.         4.70588235 0.        ]
 [0.         0.         1.        ]]
Pixel size: 0.2125 µm/px


## Build a categorical color map for `cell_type`

In [6]:
cell_types = adata.obs["cell_type"].astype(str).to_numpy()
categories = sorted(pd.unique(cell_types))

cmap = plt.get_cmap("tab20", len(categories))
color_cycle = [cmap(i) for i in range(len(categories))]

print("cell_type categories:", categories)

cell_type categories: ['AT1', 'AT2', 'B_cell', 'Ciliated', 'EC', 'Fibroblast', 'Macrophage', 'Mast', 'Neutrophil', 'PNECs', 'Plasma', 'SMC/MC', 'T_cytotoxic', 'Treg', 'Tumor_epithelial']


## Open napari

In [7]:
viewer = napari.Viewer()

/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),
/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


## Add the H&E image

In [8]:
he_image = sdata.images["he_image"]
he_scale_keys = sorted(he_image.keys())

he_pyramid = [
    he_image[key]["image"].transpose("y", "x", "c").data
    for key in he_scale_keys
]

# Alignment CSV is a (x, y, 1) affine mapping full-res H&E pixels -> global (morphology) pixel space.
he_align_path = PROJECT / "data/bundle/xenium_prime_5k/Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv"
he_affine_xy = pd.read_csv(he_align_path, header=None).to_numpy()

a, b, tx = he_affine_xy[0]
c, d, ty = he_affine_xy[1]

# Reorder rows/cols from (x, y) to napari's (y, x) axis convention.
he_affine_yx = np.array(
    [
        [d, c, ty],
        [b, a, tx],
        [0, 0, 1],
    ]
)

# Scale the affine's linear part and translation together so its output is in microns, matching every other layer.
he_affine_um = he_affine_yx.copy()
he_affine_um[:2, :] *= PIXEL_SIZE_UM

viewer.add_image(
    he_pyramid,
    multiscale=True,
    rgb=True,
    affine=he_affine_um,
    units=("um", "um"),
    name="H&E",
    visible=True,
)

<Image layer 'H&E' at 0x7f67249fe930>

## Add the morphology multiscale image

In [9]:
morphology = sdata.images["morphology_focus"]
scale_keys = sorted(morphology.keys())

morphology_pyramid = [morphology[key]["image"].data for key in scale_keys]
channel_names = list(morphology[scale_keys[0]]["image"].coords["c"].values)

viewer.add_image(
    morphology_pyramid,
    multiscale=True,
    channel_axis=0,
    name=channel_names,
    contrast_limits=[0, 20000],
    scale=(PIXEL_SIZE_UM, PIXEL_SIZE_UM),
    units=("um", "um"),
    visible=False,
)

[<Image layer 'DAPI' at 0x7f67245e9880>,
 <Image layer 'ATP1A1/CD45/E-Cadherin' at 0x7f6714da1af0>,
 <Image layer '18S' at 0x7f6714597350>,
 <Image layer 'AlphaSMA/Vimentin' at 0x7f67145dc080>]

## Build cell boundary polygons for every cell (filtering is done via layer visibility, not by rebuilding)

All cells are loaded up front; the selector below toggles visibility per `cell_type` without rebuilding the layer.

In [23]:
from shapely.geometry import MultiPolygon, Polygon
from shapely.validation import make_valid

boundaries = sdata.shapes["cell_boundaries"]

polygon_data = []
polygon_cell_types = []
n_skipped = 0

cell_type_lookup = pd.Series(cell_types, index=adata.obs["cell_labels"].astype(str).to_numpy())

for cell_id, geom in zip(boundaries.index, boundaries["geometry"]):
    # Self-intersecting/degenerate polygons crash vispy's triangulator — repair or drop them.
    if not geom.is_valid:
        geom = make_valid(geom)

    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon) or geom.is_empty:
        n_skipped += 1
        continue

    coords = np.asarray(geom.exterior.coords)

    keep = np.ones(len(coords), dtype=bool)
    keep[1:] = np.any(np.diff(coords, axis=0) != 0, axis=1)
    coords = coords[keep]
    if len(coords) < 4:  # closed ring needs >= 3 unique vertices + repeated first point
        n_skipped += 1
        continue

    coords_h = np.column_stack([coords, np.ones(len(coords))])
    coords_global = (scale_matrix @ coords_h.T).T[:, :2]

    # X,Y -> Y,X for napari
    coords_yx = coords_global[:, [1, 0]]

    polygon_data.append(coords_yx)
    polygon_cell_types.append(cell_type_lookup.get(str(cell_id), "unknown"))

polygon_cell_types = np.array(polygon_cell_types)

# Include "unknown" as its own category so no polygon lookup falls back to NaN.
shape_categories = categories + ["unknown"] if "unknown" in polygon_cell_types else categories
shape_color_cycle = color_cycle + [(0.5, 0.5, 0.5, 1.0)] if "unknown" in polygon_cell_types else color_cycle

print("Polygons:", len(polygon_data))
print("Skipped (invalid/degenerate):", n_skipped)
print("Unmatched (unknown):", (polygon_cell_types == "unknown").sum())

Polygons: 278206
Skipped (invalid/degenerate): 122
Unmatched (unknown): 12144


In [24]:
shapes_layer = viewer.add_shapes(
    polygon_data,
    shape_type="polygon",
    properties={"cell_type": pd.Categorical(polygon_cell_types, categories=shape_categories)},
    edge_width=0,
    face_color="cell_type",
    face_color_cycle=shape_color_cycle,
    scale=(PIXEL_SIZE_UM, PIXEL_SIZE_UM),
    units=("um", "um"),
    name=f"Cell boundaries — {len(polygon_data):,}",
    visible=True,
)

## Cell type of interest selector

A checklist dock widget: only the checked `cell_type`s remain visible in the boundaries layer, via the shapes layer's per-shape `shown` mask (no rebuilding of the layer).

In [ ]:
from magicgui.widgets import CheckBox, Container, create_widget

print("Shapes in layer:", len(shapes_layer.data), "| polygon_cell_types:", len(polygon_cell_types))

category_rgba = {category: np.array(rgba) for category, rgba in zip(shape_categories, shape_color_cycle)}
base_face_color = np.array([category_rgba[ct] for ct in polygon_cell_types])


def _to_hex(rgba):
    r, g, b = (int(round(c * 255)) for c in rgba[:3])
    return f"#{r:02x}{g:02x}{b:02x}"


checkboxes = [CheckBox(name=category, text=category, value=False) for category in shape_categories]
for cb, category in zip(checkboxes, shape_categories):
    color_hex = _to_hex(category_rgba[category])
    cb.native.setStyleSheet(
        f"QCheckBox::indicator {{ background-color: {color_hex}; border: 1px solid #888; }}"
    )


def _update_shown(event=None):
    selected = {cb.name for cb in checkboxes if cb.value}
    hidden = ~np.isin(polygon_cell_types, list(selected))

    new_face_color = base_face_color.copy()
    new_face_color[hidden, 3] = 0.0

    print("Selected:", selected, "| hidden shapes:", hidden.sum())
    shapes_layer.face_color = new_face_color
    shapes_layer.refresh()


for cb in checkboxes:
    cb.changed.connect(_update_shown)

cell_type_container = Container(widgets=checkboxes, labels=False, label="Cell types")
viewer.window.add_dock_widget(cell_type_container, area="right", name="Cell type of interest")
_update_shown()

Shapes in layer: 278206 | polygon_cell_types: 278206
Selected: set() | hidden shapes: 278206


Selected: {'Ciliated'} | hidden shapes: 267039
Selected: {'Macrophage', 'Ciliated'} | hidden shapes: 241076


## Show the scale bar in microns

In [13]:
viewer.scale_bar.visible = True

## Hover tooltip showing `cell_type`

In [27]:
import time

viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 12

# get_value() on this many polygons is expensive; only run it a few times per second.
TOOLTIP_INTERVAL_S = 0.08
_last_tooltip_time = 0.0


@shapes_layer.mouse_move_callbacks.append
def show_cell_type_tooltip(layer, event):
    global _last_tooltip_time
    now = time.monotonic()
    if now - _last_tooltip_time < TOOLTIP_INTERVAL_S:
        return
    _last_tooltip_time = now

    shape_index = layer.get_value(
        event.position,
        view_direction=event.view_direction,
        dims_displayed=event.dims_displayed,
        world=True,
    )
    if isinstance(shape_index, tuple):
        shape_index = shape_index[0]

    if shape_index is not None:
        cell_type = layer.properties["cell_type"][shape_index]
        viewer.text_overlay.text = f"cell_type: {cell_type}"
    else:
        viewer.text_overlay.text = ""

## Overview / minimap panel (like Xenium Explorer's bottom-left thumbnail)

Downsampled thumbnail of the whole sample with a rectangle tracking the current camera view.

In [28]:
import matplotlib
matplotlib.use("QtAgg")
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from qtpy.QtCore import QTimer

# Re-running this cell must not stack duplicate dock widgets / camera callbacks.
if "_overview_update_cb" in globals():
    viewer.scene.camera.events.center.disconnect(_overview_update_cb)
    viewer.scene.camera.events.zoom.disconnect(_overview_update_cb)
if "overview_dock_widget" in globals():
    viewer.window.remove_dock_widget(overview_dock_widget)

# Coarsest morphology pyramid level as the overview thumbnail.
overview_level = np.asarray(morphology_pyramid[-1][0])
downsample = morphology_pyramid[0].shape[-1] / morphology_pyramid[-1].shape[-1]
overview_width_um = morphology_pyramid[-1].shape[-1] * downsample * PIXEL_SIZE_UM
overview_height_um = morphology_pyramid[-1].shape[-2] * downsample * PIXEL_SIZE_UM

overview_fig = Figure(figsize=(3, 3))
overview_ax = overview_fig.add_subplot(111)
overview_ax.imshow(overview_level, cmap="gray", extent=[0, overview_width_um, overview_height_um, 0])
overview_ax.set_xticks([])
overview_ax.set_yticks([])
overview_fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

view_rect = Rectangle((0, 0), 1, 1, edgecolor="yellow", facecolor="none", linewidth=1.5)
overview_ax.add_patch(view_rect)

overview_canvas = FigureCanvasQTAgg(overview_fig)
overview_canvas.draw()


def get_canvas_size():
    qt_viewer = getattr(viewer.window, "_qt_viewer", None) or viewer.window.qt_viewer
    size = qt_viewer.canvas.size
    return size[0], size[1]


def _redraw_overview():
    overview_canvas.draw_idle()


# Camera events fire continuously while panning/zooming; debounce with a single-shot Qt timer
# so we redraw at most a few times per second instead of on every event.
_overview_timer = QTimer()
_overview_timer.setSingleShot(True)
_overview_timer.setInterval(30)
_overview_timer.timeout.connect(_redraw_overview)


def _overview_update_cb(event=None):
    camera = viewer.scene.camera
    width_px, height_px = get_canvas_size()
    zoom = camera.zoom
    center_y, center_x = camera.center[-2:]

    half_width = width_px / zoom / 2
    half_height = height_px / zoom / 2

    view_rect.set_xy((center_x - half_width, center_y - half_height))
    view_rect.set_width(2 * half_width)
    view_rect.set_height(2 * half_height)

    _overview_timer.start()


viewer.scene.camera.events.center.connect(_overview_update_cb)
viewer.scene.camera.events.zoom.connect(_overview_update_cb)
_overview_update_cb()
_redraw_overview()

overview_dock_widget = viewer.window.add_dock_widget(overview_canvas, area="left", name="Overview")

Selected: {'Macrophage', 'Treg', 'Ciliated'} | hidden shapes: 209383
Selected: {'Macrophage', 'Treg', 'Ciliated', 'T_cytotoxic'} | hidden shapes: 199138
Selected: {'Fibroblast', 'Treg', 'T_cytotoxic', 'Macrophage', 'Ciliated'} | hidden shapes: 165484
Selected: {'Fibroblast', 'Treg', 'T_cytotoxic', 'Macrophage', 'Ciliated', 'Tumor_epithelial'} | hidden shapes: 126881
Selected: {'Fibroblast', 'Treg', 'T_cytotoxic', 'Macrophage', 'Tumor_epithelial'} | hidden shapes: 138048
Selected: {'Tumor_epithelial', 'Fibroblast', 'Treg', 'T_cytotoxic'} | hidden shapes: 164011
Selected: {'Tumor_epithelial', 'Fibroblast', 'T_cytotoxic'} | hidden shapes: 195704
Selected: {'Fibroblast', 'Tumor_epithelial'} | hidden shapes: 205949
Selected: {'Tumor_epithelial', 'Fibroblast', 'T_cytotoxic'} | hidden shapes: 195704
Selected: {'Tumor_epithelial', 'Fibroblast', 'Treg', 'T_cytotoxic'} | hidden shapes: 164011
Selected: {'Fibroblast', 'Treg', 'Tumor_epithelial'} | hidden shapes: 174256
Selected: {'Fibroblast', 'T